<a href="https://colab.research.google.com/github/AmorimSilva/Codigos-Gerais/blob/main/GEE_ET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install geemap earthengine-API

In [2]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import time

In [3]:
ee.Authenticate()
ee.Initialize(project='zinc-fusion-290009')

In [43]:
# ====================================================================================
# CÉLULA 2: DEFINIÇÃO DA ROI E COMPOSIÇÃO DA SÉRIE TEMPORAL (CORRIGIDA)
# ====================================================================================


# 1. Definição da Região de Interesse (ROI)
parelhas = ee.FeatureCollection('FAO/GAUL/2015/level2') \
  .filter(ee.Filter.equals('ADM0_NAME', 'Brazil')) \
  .filter(ee.Filter.equals('ADM1_NAME', 'Rio Grande Do Norte')) \
  .filter(ee.Filter.equals('ADM2_NAME', 'Parelhas'))

# 2. Carrega as 2 coleções de dados necessárias
c1_v5 = ee.ImageCollection("MODIS/NTSG/MOD16A2/105").select(['ET', 'ET_QC'])  # Apenas para o ano 2000
c2_v61 = ee.ImageCollection("MODIS/061/MOD16A2").select(['ET', 'ET_QC'])     # Para 2001 em diante

# 3. Função de harmonização com a correção .toFloat()
def harmonize_modis_et(image):
  qc = image.select('ET_QC')
  good_quality = qc.bitwiseAnd(1).eq(0)
  et = image.select('ET').multiply(0.1).toFloat().updateMask(good_quality)
  return et.rename('ET').copyProperties(image, ['system:time_start'])

# 4. Aplica a harmonização e filtra cada coleção para seu período
et1 = c1_v5.filterDate('2000-01-01', '2000-12-31').map(harmonize_modis_et)
et2 = c2_v61.filterDate('2001-01-01', '2025-12-31').map(harmonize_modis_et)

# 5. Une as 2 coleções em uma única série temporal
ET_harmonized = ee.ImageCollection(et1.merge(et2))

print('Coleção Final Simplificada e Combinada:', ET_harmonized.size().getInfo(), 'imagens')



Coleção Final Simplificada e Combinada: 255 imagens


In [44]:
# Define o período de análise
startyear_analise = 2000
endyear_analise = 2024
years = ee.List.sequence(startyear_analise, endyear_analise)
years = years.slice(1);
months = ee.List.sequence(1, 12)

In [45]:
# ====================================================================================
#  ANÁLISE ANUAL E MENSAL
# ====================================================================================

# CORREÇÃO: A linha "years.slice(1)" foi removida para analisar o período completo.
years = ee.List.sequence(startyear_analise, endyear_analise)
months = ee.List.sequence(1, 12)

# --- Análise Anual Corrigida ---
def calculate_annual_et(year):
    annual_collection = ET_harmonized.filter(ee.Filter.calendarRange(year, year, 'year'))
    # CORREÇÃO: Retorna 'None' para anos sem dados para não distorcer os resultados.
    return ee.Algorithms.If(
        annual_collection.size().gt(0),
        annual_collection.mean().rename('ET').set('year', year).set('system:time_start', ee.Date.fromYMD(year, 1, 1)),
        None
    )

# Cria a coleção anual, removendo os anos nulos.
annual_et = ee.ImageCollection.fromImages(years.map(calculate_annual_et).removeAll([None]))

# --- Análise Mensal Corrigida ---
def calculate_monthly_et(year):
    def calculate_monthly(month):
        monthly_collection = ET_harmonized \
            .filter(ee.Filter.calendarRange(year, year, 'year')) \
            .filter(ee.Filter.calendarRange(month, month, 'month'))
        # CORREÇÃO: Retorna 'None' para meses sem dados.
        return ee.Algorithms.If(
            monthly_collection.size().gt(0),
            monthly_collection.mean().rename('ET').set('year', year).set('month', month).set('system:time_start', ee.Date.fromYMD(year, month, 1)),
            None
        )
    return months.map(calculate_monthly)

# Cria a coleção mensal, removendo os meses nulos e achatando a lista.
monthly_et = ee.ImageCollection.fromImages(years.map(calculate_monthly_et).flatten().removeAll([None]))

print("Análises anual e mensal robustas concluídas.")



Análises anual e mensal robustas concluídas.


In [59]:
# ====================================================================================
# CÉLULA 5: VISUALIZAÇÃO E CÁLCULO DA MÉDIA TOTAL
# ====================================================================================

# Cria o mapa centrado em Parelhas
Map = geemap.Map(center=[-5.9, -36.6], zoom=9)

# Calcula a imagem de média plurianual
et_vis = annual_et.mean().clip(parelhas)

# Define os parâmetros de visualização
vis_params = {
  'min': 0,
  'max': 15,
  'palette': ['blue', 'green', 'yellow', 'red']
}

# Adiciona as camadas ao mapa
Map.addLayer(parelhas, {'color': 'blue'}, 'Município de Parelhas')
Map.addLayer(et_vis, vis_params, 'ET Média Plurianual')

# Adiciona a legenda e o controle de camadas
Map.add_colorbar(vis_params, label="ET Média (kg/m²/8 dias)")
Map.add_layer_control()

# Calcula o valor numérico da média total
media_total_dict = et_vis.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=parelhas.geometry(),
    scale=500,
    maxPixels=1e9
)
media_total_valor = media_total_dict.get('ET')
print(f"A Evapotranspiração Média Total para Parelhas é: {media_total_valor.getInfo():.2f} kg/m²/8 dias")

# Mostra o mapa
Map



A Evapotranspiração Média Total para Parelhas é: 12.52 kg/m²/8 dias


Map(center=[-5.9, -36.6], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…

In [58]:
# ====================================================================================
# CÉLULA 5.2: EXPORTAR O MAPA DA MÉDIA TOTAL COMO GEOTIFF
# ====================================================================================

# A imagem 'et_vis' foi criada na Célula 5.
# Ela representa a média plurianual de ET para Parelhas.

print("Iniciando a criação da tarefa para exportar a Média Total...")

# Cria a tarefa de exportação para a imagem da média total.
task_media_total = ee.batch.Export.image.toDrive(
    image=et_vis,  # A imagem da média plurianual que já foi calculada
    description='ET_Media_Total_Parelhas_2000-2024', # Nome do arquivo
    folder='GEE_Exports_ET_Total', # Pasta de destino no seu Google Drive
    scale=500,
    region=parelhas.geometry(), # Define a área de corte
    fileFormat='GeoTIFF'
)

# Inicia a tarefa de exportação.
task_media_total.start()

# Imprime uma mensagem de confirmação.
# Você pode monitorar o progresso na aba 'Tasks' do Google Earth Engine Code Editor
# ou esperar que o arquivo apareça na pasta do seu Google Drive.
print("Tarefa de exportação da Média Total enviada.")
print("Verifique o progresso na aba 'Tasks' do GEE ou no seu Google Drive.")

# (Opcional) Se quiser monitorar a conclusão desta tarefa específica aqui no Colab:
# monitor_ee_tasks([task_media_total])

Iniciando a criação da tarefa para exportar a Média Total...
Tarefa de exportação da Média Total enviada.
Verifique o progresso na aba 'Tasks' do GEE ou no seu Google Drive.


In [57]:
# ====================================================================================
# CÉLULA 6: EXPORTAÇÃO DE DADOS COM MONITORAMENTO AUTOMÁTICO (OPCIONAL)
# ====================================================================================

# --- Função para monitorar as tarefas ---
def monitor_ee_tasks(tasks):
    """Verifica o status de uma lista de tarefas do EE e espera a conclusão."""
    print(f"Monitorando {len(tasks)} tarefa(s)...")
    while True:
        states = [task.status()['state'] for task in tasks]
        running_tasks = states.count('RUNNING')
        ready_tasks = states.count('READY')
        completed_tasks = states.count('COMPLETED')
        failed_tasks = states.count('FAILED')
        print(f"Status: {running_tasks} Rodando, {ready_tasks} Prontas, {completed_tasks} Concluídas, {failed_tasks} Falharam.")
        if running_tasks == 0 and ready_tasks == 0:
            print("Todas as tarefas foram concluídas!")
            break
        time.sleep(60)

# --- Lista para armazenar as tarefas criadas ---
tasks_para_monitorar = []

In [37]:
# ==============================================================================
# ATENÇÃO: Descomente UM dos blocos abaixo (anual ou mensal) para exportar.
# ==============================================================================

# --- Bloco de Exportação Anual ---
count_anual = annual_et.size().getInfo()
image_list_anual = annual_et.toList(count_anual)
print(f'Iniciando a criação de {count_anual} tarefas de exportação ANUAL...')
for i in range(count_anual):
    image = ee.Image(image_list_anual.get(i))
    year = image.get('year').getInfo()
    file_name = f'ET_Anual_Parelhas_{year:04d}'
    task = ee.batch.Export.image.toDrive(image=image.clip(parelhas.geometry()), description=file_name, folder='GEE_Exports_ET_Anual_Completo_2', scale=500, region=parelhas.geometry(), fileFormat='GeoTIFF')
    task.start()
    tasks_para_monitorar.append(task)
print("Criação de tarefas anuais concluída.")


# --- Bloco de Exportação Mensal ---
# count_mensal = monthly_et.size().getInfo()
# image_list_mensal = monthly_et.toList(count_mensal)
# print(f'Iniciando a criação de {count_mensal} tarefas de exportação MENSAL...')
# for i in range(count_mensal):
#     image = ee.Image(image_list_mensal.get(i))
#     year = image.get('year').getInfo()
#     month = image.get('month').getInfo()
#     file_name = f'ET_Mensal_Parelhas_{year:04d}_{month:02d}'
#     task = ee.batch.Export.image.toDrive(image=image.clip(parelhas.geometry()), description=file_name, folder='GEE_Exports_ET_Mensal_Completo', scale=500, region=pare

Iniciando a criação de 23 tarefas de exportação ANUAL...


KeyboardInterrupt: 